In [ ]:
import warnings
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("656790491", h="6325554e55", width=600)


In [ ]:
def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400_000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    # Split "lat-lon" column
    df[["lat", "lon"]] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon", inplace=True)

    # Extract neighbor place
    df["neighborhood"] = df["place_with_parent_names"].str.split("|", expand=True)[3]
    df.drop(columns="place_with_parent_names", inplace=True)
    
    return df


In [ ]:
VimeoVideo("656790237", h="1502e3765a", width=600)


In [ ]:
files = glob("data/buenos-aires-real-estate-*.csv")
files


In [ ]:
# Check your work
assert len(files) == 5, f"`files` should contain 5 items, not {len(files)}"


In [ ]:
VimeoVideo("656789768", h="3b8f3bca0b", width=600)


In [ ]:
frames = []
for file in files:
    df = wrangle(file)
    frames.append(df)


In [ ]:
# Check your work
assert len(frames) == 5, f"`frames` should contain 5 items, not {len(frames)}"
assert all(
    [isinstance(frame, pd.DataFrame) for frame in frames]
), "The items in `frames` should all be DataFrames."


In [ ]:
VimeoVideo("656789700", h="57adef4afe", width=600)


In [ ]:
df = pd.concat(frames, ignore_index=True)
df.shape


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` is the wrong size: {len(df)}."


In [ ]:
VimeoVideo("656791659", h="581201dc92", width=600)


In [ ]:
# Check your work
assert df.shape == (6582, 17), f"`df` is the wrong size: {df.shape}."
assert (
    "place_with_parent_names" not in df
), 'Remember to remove the `"place_with_parent_names"` column.'


In [ ]:
VimeoVideo("656791577", h="0ceb5341f8", width=600)


In [ ]:
target = "price_aprox_usd"
features = ["neighborhood"]
y_train = df[target]
X_train = df[features]


In [ ]:
# Check your work
assert X_train.shape == (6582, 1), f"`X_train` is the wrong size: {X_train.shape}."
assert y_train.shape == (6582,), f"`y_train` is the wrong size: {y_train.shape}."


In [ ]:
VimeoVideo("656791443", h="120a740cc3", width=600)


In [ ]:
y_mean = y_train.mean()
y_pred_baseline = [y_mean] * len(y_train)

print("Mean apt price:", y_mean)
print("Baseline MAE:", mean_absolute_error(y_train,y_pred_baseline))


In [ ]:
VimeoVideo("656792790", h="4097efb40d", width=600)


In [ ]:
#Instantiate
ohe = OneHotEncoder(use_cat_names=True)
# fit
ohe.fit(X_train)
#Transform
XT_train = ohe.transform(X_train)
print(XT_train.shape)
XT_train.head()


In [ ]:
# Check your work
assert XT_train.shape == (6582, 57), f"`XT_train` is the wrong shape: {XT_train.shape}"


In [ ]:
VimeoVideo("656792622", h="0b9d189e8f", width=600)


In [ ]:
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    Ridge()
)
model.fit(X_train,y_train)


In [ ]:
# Check your work
check_is_fitted(model[-1])


In [ ]:
VimeoVideo("656792525", h="09edc1c3d6", width=600)


In [ ]:
y_pred_training = model.predict(X_train)
mae_training = mean_absolute_error(y_train, y_pred_training)
print("Training MAE:", round(mae_training, 2))


In [ ]:
X_test = pd.read_csv("data/buenos-aires-test-features.csv")[features]
y_pred_test = pd.Series(model.predict(X_test))
y_pred_test.head()


In [ ]:
VimeoVideo("656793909", h="fca67856b4", width=600)


In [ ]:
intercept = model.named_steps["ridge"].intercept_
coefficients = model.named_steps["ridge"].coef_
print("coefficients len:", len(coefficients))
print(coefficients[:5])  # First five coefficients


In [ ]:
# Check your work
assert isinstance(
    intercept, float
), f"`intercept` should be a `float`, not {type(intercept)}."
assert isinstance(
    coefficients, np.ndarray
), f"`coefficients` should be a `float`, not {type(coefficients)}."
assert coefficients.shape == (
    57,
), f"`coefficients` is wrong shape: {coefficients.shape}."


In [ ]:
VimeoVideo("656793812", h="810161b84e", width=600)


In [ ]:
feature_names = model.named_steps["onehotencoder"].get_feature_names()
print("features len:", len(feature_names))
print(feature_names[:5])  # First five feature names


In [ ]:
# Check your work
assert isinstance(
    feature_names, np.ndarray
), f"`features` should be a `list`, not {type(feature_names)}."
assert len(feature_names) == len(
    coefficients
), "You should have the same number of features and coefficients."


In [ ]:
VimeoVideo("656793718", h="1e2a1e1de8", width=600)


In [ ]:
feat_imp = pd.Series(coefficients, index=feature_names)
feat_imp.head()


In [ ]:
# Check your work
assert isinstance(
    feat_imp, pd.Series
), f"`feat_imp` should be a `float`, not {type(feat_imp)}."
assert feat_imp.shape == (57,), f"`feat_imp` is wrong shape: {feat_imp.shape}."
assert all(
    a == b for a, b in zip(sorted(feature_names), sorted(feat_imp.index))
), "The index of `feat_imp` should be identical to `features`."


In [ ]:
VimeoVideo("656797021", h="dc90e6dac3", width=600)


In [ ]:
print(f"price = {intercept.round(2)}")
for f, c in feat_imp.items():
    print(f"+ ({round(c, 2)} * {f})")


In [ ]:
VimeoVideo("656799309", h="a7130deb64", width=600)


In [ ]:
# Check your work
assert isinstance(
    model[-1], Ridge
), "Did you retrain your model using a `Ridge` predictor?"


In [ ]:
VimeoVideo("656798530", h="9a9350eff1", width=600)


In [ ]:
feat_imp.sort_values(key=abs).tail(15).plot(kind="barh")
plt.xlabel("importance [USD]")
plt.ylabel("Feature")
plt.title("Feature importance for Apartment price");
